- 예외 설계

우리 프로젝트에서 예외 부모

In [ ]:
class AgentError(Exception):
    status_code = 400
    code = "agent_error"

    def __init__(self, message: str, *, detail: str | None = None):
        # print(e) 했을 때 메시지가 나오게 하기 위해 Exception의 message를 가져온다.
        super().__init__(message) 
        self.message = message
        self.detail = detail

자식 클래스로 세분화

In [ ]:
# 요청한 자원이 없다
class NotFound(AgentError):
    status_code = 404
    code = "not_found"

# 자원은 있으나, 이 사용자가 접근할 수 없다.
class PermissionDenied(AgentError):
    status_code = 403
    code = "permission_denied"

# 입력값이 규칙에 맞지 않다.
class ValidationError(AgentError):
    status_code = 422
    code = "validation_error"

# 외부 서비스 호출 실패
class ExternalServiceError(AgentError):
    status_code = 502
    code = "external_service_error"

e = NotFound("문서를 찾을 수 없습니다.", detail="doc_id에 id가 없는 값")
print(f"메시지 : {e}")
print(f"상태코드 : {e.status_code}")
print(f"코드 : {e.code}")
print(f"상세 : {e.detail}")

메시지 : 문서를 찾을 수 없습니다.
상태코드 : 404
코드 : not_found
상세 : doc_id에 id가 없는 값


*사용 예시*

In [5]:
def handle(exc):
    if isinstance(exc, AgentError):
        # 우리가 의도한 상황
        return {"message": exc.message,"status": exc.status_code, "code": exc.code, "detail": exc.detail}
    # 우리가 예상하지 못한 예외 발생 (진짜 사고)
    return {"message": "서버 오류가 발생했습니다.", "status": 500, "code": "internal_error"}

for exc in [
    NotFound("문서를 못 찾았습니다."),
    PermissionDenied("이 문서를 열람할 권한이 없습니다."),
    ExternalServiceError("문서 변환 서비스에 연결하지 못했습니다."),
    KeyError("doc_id")
]:
    print(handle(exc))

{'message': '문서를 못 찾았습니다.', 'status': 404, 'code': 'not_found', 'detail': None}
{'message': '이 문서를 열람할 권한이 없습니다.', 'status': 403, 'code': 'Permission_denied', 'detail': None}
{'message': '문서 변환 서비스에 연결하지 못했습니다.', 'status': 502, 'code': 'external_service_error', 'detail': None}
{'message': '서버 오류가 발생했습니다.', 'status': 500, 'code': 'internal_error'}


- raise...from...

In [ ]:
import json

def parse_bad(text):
    try:
        # 문자열을 파이썬 객체로 변환
        return json.loads(text)
    except json.JSONDecodeError as e:
        # 우리가 설계한 예외 클래스로 예외 발생시키기
        raise ValidationError(
            "데이터 형식이 올바르지 않습니다.", # 우리가 설계한 예외 메시지 추가 (사용자용)
            detail=str(e) # 원래 터진 에러 메시지 추가 (개발자용)
        ) from e # 원인 연결

try:
    parse_bad("{json 형식이 아닌 input}")
except ValidationError as e:
    print("예외 메시지 : ", e)
    print("원인 :", type(e.__cause__).__name__, e.__cause__)


예외 메시지 :  데이터 형식이 올바르지 않습니다.
원인 : JSONDecodeError   Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
